# Quoridor AI — 9×9 Training (N=2)

**Group 501** | Colman College | DL Final Project

Full-size Quoridor (9×9, 10 walls/player) trained via sequential self-play.
All hyperparameters are read from `configs/config_9x9.json`.

**Two ways to run:**

**A. Interactively** (this notebook) — run cells top to bottom.
If the kernel disconnects, re-run from Section 1; `resume=True` picks up from the last checkpoint.

**B. Detached / headless** (survives SSH or browser disconnect) — run from a terminal:
```bash
scripts/run_notebook.sh                                  # this notebook (N=2)
NOTEBOOK=notebooks/train_9x9_n4.ipynb scripts/run_notebook.sh   # N=4
tail -f runs/train_9x9_n2/notebook.log                   # monitor
```
This executes the whole notebook via `nohup jupyter nbconvert --execute`, so it
keeps running at the OS level even if you close the browser or drop SSH.

## Security & Data Warning

**DO NOT commit or upload:**
- API keys, authentication tokens, or passwords
- Model checkpoints containing sensitive training data
- File paths with personal/org information

**Best practices:**
- Store checkpoints in private cloud storage or local machine
- Use `.gitignore` to exclude model files: `*.pth`, `*.ckpt`, `configs/*.local`
- For Colab: download checkpoints before session ends or save to private Drive

## Expected Output & Monitoring

**During training, expect:**
- `[GPU] 500 batches processed...` every ~500 batches (progress indicator)
- Game summaries: `winner=Player X, walls: [8, 9, 8, 10]` (one per iteration)
- Win statistics at iteration end: `{0: 45, 1: 35, 2: 10, None: 10}` (draws)
- Loss/accuracy metrics from model training

**Runtime expectations (9x9, N=2, T4 GPU):**
- ~8 min per self-play iteration (200 games with batched inference)
- ~2 min model training per iteration
- Full training (100 iterations): ~16-20 hours

**Troubleshooting:**
- **OOM error**: Reduce `inference_batch_size` in config or `num_workers`
- **Slow training**: Ensure GPU is active (check `!nvidia-smi`)
- **Hung process**: Cells timeout after 90 min; re-run will resume with `resume=True`

---
## 1. Environment Setup
Run once per Colab session.

In [ ]:
# 1.1 — Install dependencies and setup repo
import os, sys

# Find repo root (works from any starting directory)
REPO_DIR = None
for candidate in [
    os.getcwd(),                             # already in repo root
    os.path.join(os.getcwd(), "dl-quoridor"),  # Jupyter workspace root
    os.path.dirname(os.getcwd()),            # running from notebooks/
]:
    if os.path.exists(os.path.join(candidate, ".git")):
        REPO_DIR = candidate
        break

assert REPO_DIR is not None, (
    "Could not find dl-quoridor repo. "
    "Make sure the notebook is inside the repo or one level above it."
)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Ensure we're on the latest dev branch
BRANCH = "dev"
!cd {REPO_DIR} && git checkout . && git checkout {BRANCH} && git pull

print(f"Repo: {REPO_DIR}")

# Install requirements (--ignore-installed handles system-managed packages)
!pip install -r requirements.txt -q --ignore-installed
print("Dependencies installed.")

In [ ]:
# 1.2 — Detect hardware and auto-tune parallel settings
import torch, multiprocessing, psutil

print(f"torch {torch.__version__}")
CPU_CORES = multiprocessing.cpu_count()
RAM_GB = psutil.virtual_memory().total / 1e9
print(f"CPU cores: {CPU_CORES}")
print(f"RAM: {RAM_GB:.1f} GB")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected — training will run on CPU (slow for 9x9).")

# Auto-tune parallel workers based on available resources
# 16 workers = ~935 evals/s on RTX PRO 6000 (measured). 50 was slower (queue overhead).
# Try 32 if you want to experiment — might be slightly better or worse.
# NOTE: run only ONE training notebook at a time — GPU context-switching
#       between two kernels kills throughput for the slower one.
AUTO_WORKERS = 32
# Batch size: scale with VRAM (model is tiny, ~10MB)
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    AUTO_BATCH = 64 if vram_gb < 8 else 128 if vram_gb < 16 else 256
else:
    AUTO_BATCH = 32
print(f"\n→ Auto-tuned: {AUTO_WORKERS} workers, batch={AUTO_BATCH}")

In [ ]:
# 1.3 — Set run directory for checkpoints
RUN_DIR = os.path.join(REPO_DIR, "runs", "n2_9x9_v2")
os.makedirs(RUN_DIR, exist_ok=True)
print(f"Run dir: {RUN_DIR}")

---
## 2. Configuration

In [ ]:
# 2.1 — Load config from configs/config_9x9.json
import json

VARIANT = "n2"  # ← change to "n4" for 4-player training

CONFIG_PATH = f"{REPO_DIR}/configs/config_9x9.json"
with open(CONFIG_PATH) as f:
    cfg = json.load(f)

# Merge variant-specific overrides into top level
variant = cfg["variants"][VARIANT]

N = variant["num_players"]
BOARD = cfg["board_size"]
WALLS = variant["max_walls_per_player"]
MAX_TURNS = cfg["mcts"]["max_rollout_depth"]

# Network
NUM_CHANNELS = cfg["network"]["num_channels"]
NUM_RES_BLOCKS = cfg["network"]["num_res_blocks"]
IN_CHANNELS = 3 * N + 3

# Action space
from src.env.quoridor_env_mp import compute_action_space_size
ACTION_SIZE = compute_action_space_size(BOARD)

# Training
NUM_ITERATIONS = cfg["training"]["num_iterations"]
GAMES_PER_ITER = cfg["training"]["games_per_iteration"]
MCTS_SIMS = cfg["mcts"]["num_simulations"]
EVAL_SIMS = cfg["mcts"].get("eval_simulations", 0)
BATCH_SIZE = cfg["training"]["batch_size"]
TRAIN_STEPS = cfg["training"]["training_epochs"]
REPLAY_BUFFER = cfg["training"]["replay_buffer_size"]
from src.utils.config import variant_setting
# Per-variant: max_game_moves counts plies, so one shared value gives N=4
# players half the per-player budget of N=2 players on the same board.
MAX_MOVES = variant_setting(cfg, VARIANT, "max_game_moves")
DISCOUNT = cfg["training"]["reward_decay"]
LR = cfg["training"]["learning_rate"]
WEIGHT_DECAY = cfg["training"]["weight_decay"]
ACCEPT_MARGIN = cfg["training"].get("accept_margin", 0.05)
EVAL_EVERY = cfg["training"].get("eval_every", 1)
EXPLORE_MOVES = cfg["training"].get("explore_moves", 15)
WARMUP_MIN_SAMPLES = cfg["training"].get("warmup_min_samples", 0)
EVAL_GAMES = variant["eval_games"]
EVAL_RANDOM = variant["eval_random_games"]

# Parallel: workers/batch auto-tuned from hardware; toggles from config
NUM_WORKERS = AUTO_WORKERS
INFER_BATCH = AUTO_BATCH
PARALLEL_SELF_PLAY = cfg["parallel"].get("parallel_self_play", True)
PARALLEL_EVAL = cfg["parallel"].get("parallel_eval", False)
# Leaf-parallel MCTS: leaves collected per GPU forward + virtual loss (1 = one-leaf path)
LEAF_BATCH = cfg["parallel"].get("leaf_batch", 1)
VIRTUAL_LOSS = cfg["parallel"].get("virtual_loss", 1.0)
# Self-play engine: 'auto'|'sequential'|'parallel'|'vectorized' (Option B in-process)
SELF_PLAY_MODE = cfg["parallel"].get("self_play_mode", "auto")
VEC_GAMES = cfg["parallel"].get("vec_games", 0)

print(f"Loaded: {CONFIG_PATH} [variant={VARIANT}]")
print(f"Config: N={N}, board={BOARD}×{BOARD}, actions={ACTION_SIZE}, walls={WALLS}")
print(f"Network: {NUM_RES_BLOCKS} res blocks, {NUM_CHANNELS} channels, {IN_CHANNELS} input planes")
print(f"Training: {NUM_ITERATIONS} iters, {GAMES_PER_ITER} games/iter, {MCTS_SIMS} sims, "
      f"{TRAIN_STEPS} train steps, max_moves={MAX_MOVES}, explore={EXPLORE_MOVES}, warmup={WARMUP_MIN_SAMPLES}")
print(f"Parallel: {NUM_WORKERS} workers, batch={INFER_BATCH}, leaf_batch={LEAF_BATCH}, vloss={VIRTUAL_LOSS} "
      f"(auto-tuned from {CPU_CORES} cores) | self_play={PARALLEL_SELF_PLAY} eval={PARALLEL_EVAL}")

---
## 3. Validation
Quick smoke test before committing hours of GPU time.

In [ ]:
# 3.1 — Smoke test: env + tensor + model forward pass at 9×9
import numpy as np
from src.env.quoridor_env_mp import QuoridorEnvMP
from src.model.network_mp import QuoridorModelMP

env = QuoridorEnvMP(board_size=BOARD, num_players=N, max_turns=MAX_TURNS,
                    max_walls_per_player=WALLS)
state = env.reset()
tensor = env.state_to_tensor(state)

print(f"Tensor shape: {tensor.shape}  (expected: ({BOARD}, {BOARD}, {IN_CHANNELS}))")
assert tensor.shape == (BOARD, BOARD, IN_CHANNELS)

model = QuoridorModelMP(
    board_size=BOARD, action_space_size=ACTION_SIZE,
    in_channels=IN_CHANNELS, num_channels=NUM_CHANNELS,
    num_res_blocks=NUM_RES_BLOCKS, num_players=N,
    lr=LR, weight_decay=WEIGHT_DECAY,
)

policy, value = model.predict(tensor)
print(f"Policy shape: {policy.shape}  (expected: ({ACTION_SIZE},))")
print(f"Value shape:  {value.shape}  (expected: ({N},))")
assert policy.shape == (ACTION_SIZE,)
assert value.shape == (N,)

# Batch forward pass (what the GPU inference thread does)
batch = torch.from_numpy(tensor).float().permute(2, 0, 1).unsqueeze(0).repeat(INFER_BATCH, 1, 1, 1).to(model.device)
policies, values = model.predict_batch(batch)
print(f"Batch policy: {policies.shape}  (expected: ({INFER_BATCH}, {ACTION_SIZE}))")
print(f"Batch value:  {values.shape}  (expected: ({INFER_BATCH}, {N}))")

params = sum(p.numel() for p in model.network.parameters())
print(f"\nModel parameters: {params:,}")
print("\n✓ Smoke test passed — 9×9 pipeline is connected.")

In [ ]:
# 3.2 — Quick random game at 9×9 (no crash test)
# (reuses env from cell above)
state = env.reset()
moves = 0
while not state.game_over:
    actions = env.get_valid_actions(state)
    action = np.random.choice(actions)
    state, _, done, _ = env.step(state, action)
    moves += 1

winner = f"P{state.winner}" if state.winner is not None else "Draw (turn limit)"
print(f"Random game finished in {moves} moves. Winner: {winner}")
print("✓ No crashes during random play.")

---
## 4. Training
Re-run this cell after Colab disconnects — `resume=True` picks up from the last checkpoint.

In [ ]:
# 4 — Full 9×9 training (parallel self-play, GPU-batched)
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    force=True,
)

from src.env.quoridor_env_mp import QuoridorEnvMP
from src.model.network_mp import QuoridorModelMP
from src.mcts.training_mp import TrainingConfigMP, training_loop_mp

env = QuoridorEnvMP(board_size=BOARD, num_players=N, max_turns=MAX_TURNS,
                    max_walls_per_player=WALLS)

def make_model():
    return QuoridorModelMP(
        board_size=BOARD, action_space_size=ACTION_SIZE,
        in_channels=IN_CHANNELS, num_channels=NUM_CHANNELS,
        num_res_blocks=NUM_RES_BLOCKS, num_players=N,
        lr=LR, weight_decay=WEIGHT_DECAY, device="auto",
    )

model = make_model()

train_cfg = TrainingConfigMP(
    num_players=N,
    num_iterations=NUM_ITERATIONS,
    games_per_iteration=GAMES_PER_ITER,
    mcts_simulations=MCTS_SIMS,
    eval_simulations=EVAL_SIMS,
    batch_size=BATCH_SIZE,
    train_steps_per_iter=TRAIN_STEPS,
    warmup_min_samples=WARMUP_MIN_SAMPLES,
    eval_games=EVAL_GAMES,
    eval_random_games=EVAL_RANDOM,
    accept_margin=ACCEPT_MARGIN,
    eval_every=EVAL_EVERY,
    max_game_moves=MAX_MOVES,
    explore_moves=EXPLORE_MOVES,
    discount=DISCOUNT,
    replay_buffer_size=REPLAY_BUFFER,
    mcts_dirichlet_epsilon=cfg.get('mcts', {}).get('dirichlet_epsilon', 0.25),
    # parallel self-play + GPU-batched parallel eval (toggles from config_9x9.json)
    parallel_self_play=PARALLEL_SELF_PLAY,
    parallel_eval=PARALLEL_EVAL,
    num_workers=NUM_WORKERS,
    inference_batch_size=INFER_BATCH,
    # leaf-parallel MCTS (breaks the batch<=num_workers GPU-starvation ceiling)
    leaf_batch=LEAF_BATCH,
    virtual_loss=VIRTUAL_LOSS,
    # self-play engine selector (vectorized = Option B in-process)
    self_play_mode=SELF_PLAY_MODE,
    vec_games=VEC_GAMES,
    # geometry for spawned workers
    board_size=BOARD,
    max_walls_per_player=WALLS,
    max_turns=MAX_TURNS,
)

print(f"Starting training: {RUN_DIR}")
print(f"  Resume from checkpoint if available.")

training_loop_mp(env, model, make_model, train_cfg, checkpoint_dir=RUN_DIR)

---
## 5. Monitor & Analyze Results

In [ ]:
# 5.1 — Training curves
import json
import matplotlib.pyplot as plt

meta_path = f"{RUN_DIR}/meta.json"

try:
    with open(meta_path) as f:
        meta = json.load(f)
    metrics = meta.get("history", [])
except FileNotFoundError:
    print("No meta.json yet — run training first.")
    metrics = []

if metrics:
    iters = [m['iter'] for m in metrics]
    loss_p = [m.get('loss_p', 0) for m in metrics]
    loss_v = [m.get('loss_v', 0) for m in metrics]
    wr = [m.get('win_vs_random', 0) for m in metrics]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('Quoridor 9×9 N=2 — Training Progress', fontsize=14)

    axes[0].plot(iters, loss_p, 'b-o', markersize=3)
    axes[0].set_title('Policy Loss')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Cross-Entropy')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(iters, loss_v, 'r-o', markersize=3)
    axes[1].set_title('Value Loss')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('MSE')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(iters, [w * 100 for w in wr], 'g-o', markersize=3)
    axes[2].axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
    axes[2].set_title('Win Rate vs Random')
    axes[2].set_xlabel('Iteration')
    axes[2].set_ylabel('Win Rate (%)')
    axes[2].set_ylim(0, 105)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{RUN_DIR}/training_curves.png", dpi=150)
    plt.show()

    print(f"\nLatest iteration: {iters[-1]}")
    print(f"Policy loss: {loss_p[-1]:.4f}")
    print(f"Value loss:  {loss_v[-1]:.4f}")
    print(f"Win rate:    {wr[-1]:.1%}")

In [ ]:
# 5.1b — Draw-rate trend (watchdog for weak value signal)
# A draw = a game that timed out at max_game_moves => all-zero value target
# (weak learning signal). Falling over iters = policy learning to finish games;
# stuck high past the early iters = the value head is training mostly on zeros.
import re, json
import matplotlib.pyplot as plt

draw_iters, draw_pct = [], []
# Prefer meta.json history (new runs record draw_rate); fall back to games.log
# so this also works for a run started before draw_rate was recorded.
try:
    _hist = json.load(open(f"{RUN_DIR}/meta.json")).get("history", [])
except FileNotFoundError:
    _hist = []
if any(m.get("draw_rate") is not None for m in _hist):
    for m in _hist:
        if m.get("draw_rate") is not None:
            draw_iters.append(m["iter"]); draw_pct.append(100 * m["draw_rate"])
else:
    _pat = re.compile(r">>> iter (\d+).*?draw=(\d+)%")
    try:
        for line in open(f"{RUN_DIR}/games.log"):
            g = _pat.search(line)
            if g:
                draw_iters.append(int(g.group(1))); draw_pct.append(float(g.group(2)))
    except FileNotFoundError:
        print("no meta.json history or games.log yet — run training first")

if draw_iters:
    plt.figure(figsize=(8, 4))
    plt.plot(draw_iters, draw_pct, "m-o", markersize=4)
    plt.axhline(20, color="gray", ls="--", alpha=0.6, label="20% warn threshold")
    plt.title("Draw / timeout rate per iteration")
    plt.xlabel("Iteration"); plt.ylabel("Draw rate (%)"); plt.ylim(0, 105)
    plt.legend(); plt.grid(True, alpha=0.3); plt.show()
    _tail = list(zip(draw_iters, draw_pct))[-8:]
    print("recent draw%:", ", ".join(f"i{it}:{p:.0f}%" for it, p in _tail))
    if len(draw_pct) >= 5 and draw_pct[-1] > 55:
        print("\u26a0 draw rate still high after several iters — value signal weak; "
              "consider raising max_game_moves or a progress-based timeout value.")


In [ ]:
# 5.2 — Metrics table
if metrics:
    print(f"{'Iter':>4} | {'Loss_P':>8} | {'Loss_V':>8} | {'WR_Random':>10} | {'Accepted':>8}")
    print("-" * 55)
    for m in metrics:
        print(
            f"{m['iter']:4d} | "
            f"{m.get('loss_p', 0):8.4f} | "
            f"{m.get('loss_v', 0):8.4f} | "
            f"{m.get('win_vs_random', 0):9.1%} | "
            f"{'✓' if m.get('accepted') else ''}"
        )

---
## 5.3 Run All Graph Scripts
Generate full dashboards and comparison figures (same scripts used for the project book).

In [ ]:
# 5.3 — Run all graph/plotting scripts
import subprocess, os

os.chdir(REPO_DIR)

scripts = [
    "scripts/plot_training.py",
    "scripts/plot_all_figures.py",
]

for script in scripts:
    if os.path.exists(script):
        print(f"\n{'='*60}")
        print(f"Running: {script}")
        print(f"{'='*60}")
        result = subprocess.run(
            ["python", script, RUN_DIR + "/meta.json"] if "plot_training" in script
            else ["python", script],
            capture_output=True, text=True,
            env={**os.environ, "PYTHONPATH": REPO_DIR}
        )
        if result.returncode == 0:
            print(f"✓ {script} completed")
            if result.stdout.strip():
                print(result.stdout[-500:])
        else:
            print(f"✗ {script} failed (exit {result.returncode})")
            print(result.stderr[-500:])
    else:
        print(f"⚠ {script} not found — skipping")

# Show generated figures
from IPython.display import Image, display
from pathlib import Path

fig_dirs = [
    Path(RUN_DIR) / "figures",
    Path(REPO_DIR) / "outputs",
]
for fig_dir in fig_dirs:
    if fig_dir.exists():
        for img in sorted(fig_dir.glob("*.png")):
            print(f"\n📊 {img.name}")
            display(Image(filename=str(img), width=900))

---
## 5.4 Final Evaluation at Full Sims (for the report)

Training used reduced sims (`eval_simulations`) for fast gating decisions. This
cell re-evaluates the final `best.pt` at the **full `num_simulations`** budget to
get the true strength numbers for the project book. Run once after training.

In [ ]:
# 5.4 — Final evaluation at full sims (true strength for the report)
from src.env.quoridor_env_mp import QuoridorEnvMP
from src.model.network_mp import QuoridorModelMP
from src.mcts.training_mp import _mcts, TrainingConfigMP
from src.mcts.evaluator_mp import evaluate_against_random_mp, mcts_agent_mp

FINAL_EVAL_GAMES = 100   # more games = tighter confidence interval

best_path = f"{RUN_DIR}/best.pt"
if not os.path.exists(best_path):
    print(f"No best.pt at {best_path} — train first (or no iteration was accepted yet).")
else:
    eval_env = QuoridorEnvMP(board_size=BOARD, num_players=N, max_turns=MAX_TURNS,
                             max_walls_per_player=WALLS)
    final_model = QuoridorModelMP(
        board_size=BOARD, action_space_size=ACTION_SIZE,
        in_channels=IN_CHANNELS, num_channels=NUM_CHANNELS,
        num_res_blocks=NUM_RES_BLOCKS, num_players=N, device="auto",
    )
    final_model.load(best_path)

    # Full-sim config (uses num_simulations, NOT eval_simulations)
    full_cfg = TrainingConfigMP(
        num_players=N, mcts_simulations=MCTS_SIMS,
        max_game_moves=MAX_MOVES,
        mcts_dirichlet_epsilon=cfg.get('mcts', {}).get('dirichlet_epsilon', 0.25),
    )
    agent = mcts_agent_mp(_mcts(final_model, eval_env, full_cfg), temperature=0.1)

    print(f"Evaluating best.pt at FULL {MCTS_SIMS} sims over {FINAL_EVAL_GAMES} games vs random...")
    res = evaluate_against_random_mp(eval_env, agent,
                                     num_games=FINAL_EVAL_GAMES, max_moves=MAX_MOVES)

    wr = res.candidate_win_rate
    # 95% binomial confidence interval
    import math
    se = math.sqrt(wr * (1 - wr) / FINAL_EVAL_GAMES)
    ci = 1.96 * se
    fair = 1.0 / N
    print(f"\n=== FINAL STRENGTH (full {MCTS_SIMS} sims) ===")
    print(f"Win vs random: {wr:.1%} ± {ci:.1%} (95% CI)  [fair share = {fair:.0%}]")
    print(f"Games: {FINAL_EVAL_GAMES} | {res.summary()}")

---
## 6. Export Best Model
Copy the final trained model to Drive and prepare for deployment.

In [ ]:
# 6.1 — Copy best/ship checkpoint
import shutil

best_src = f"{RUN_DIR}/best.pt"
ship_dst = f"{RUN_DIR}/ship.pt"

if os.path.exists(best_src):
    shutil.copy2(best_src, ship_dst)
    size_mb = os.path.getsize(ship_dst) / 1e6
    print(f"✓ Exported ship.pt ({size_mb:.1f} MB)")
    print(f"  Path: {ship_dst}")
else:
    print("No best.pt found — training may not have completed an accepted iteration yet.")

In [ ]:
# 6.2 — Download checkpoint to local machine (Colab only)
try:
    from google.colab import files
    if os.path.exists(ship_dst):
        files.download(ship_dst)
except ImportError:
    print("Not running on Colab — use Drive or scp to retrieve the model.")